## DM Project: Impact of Weather on US Flight Delays
This notebook performs data cleaning and quality assessment on both
the raw Kaggle flight dataset and the raw weather API dataset.

Quality dimensions evaluated: Completeness, Uniqueness, Validity, Consistency

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
import pandas as pd
import numpy as np

# Load from Raw folder
df = pd.read_csv('/content/drive/MyDrive/DM_Project/Data/Raw/flights_sample_3m.csv')
weather_df = pd.read_csv('/content/drive/MyDrive/DM_Project/Data/Raw/weather_api_raw.csv')

print("Flight dataset loaded:", df.shape)
print("Weather dataset loaded:", weather_df.shape)

Flight dataset loaded: (3000000, 32)
Weather dataset loaded: (1495, 5)


In [3]:
print("=== FLIGHT DATASET - BEFORE CLEANING ===")
print("\nMissing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)
print("\nDuplicates:", df.duplicated().sum())

print("\n=== WEATHER DATASET - BEFORE CLEANING ===")
print("\nMissing values:")
print(weather_df.isnull().sum())
print("Duplicates:", weather_df.duplicated().sum())

=== FLIGHT DATASET - BEFORE CLEANING ===

Missing values:
FL_DATE                          0
AIRLINE                          0
AIRLINE_DOT                      0
AIRLINE_CODE                     0
DOT_CODE                         0
FL_NUMBER                        0
ORIGIN                           0
ORIGIN_CITY                      0
DEST                             0
DEST_CITY                        0
CRS_DEP_TIME                     0
DEP_TIME                     77615
DEP_DELAY                    77644
TAXI_OUT                     78806
WHEELS_OFF                   78806
WHEELS_ON                    79944
TAXI_IN                      79944
CRS_ARR_TIME                     0
ARR_TIME                     79942
ARR_DELAY                    86198
CANCELLED                        0
CANCELLATION_CODE          2920860
DIVERTED                         0
CRS_ELAPSED_TIME                14
ELAPSED_TIME                 86198
AIR_TIME                     86198
DISTANCE                        

In [4]:
# Fix date format
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'], format='mixed', errors='coerce')

# Drop rows with missing critical fields
before = df.shape[0]
df = df.dropna(subset=['FL_DATE', 'AIRLINE', 'ORIGIN'])
after_drop = df.shape[0]
print(f"Rows dropped (missing critical fields): {before - after_drop}")

# Fill delay columns with 0
delay_cols = ['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER',
              'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']
df[delay_cols] = df[delay_cols].fillna(0)

# Remove extreme outliers
df = df[df['DEP_DELAY'] <= 600]
print(f"Rows after outlier removal: {df.shape[0]}")

print("\nFlight dataset cleaned:", df.shape)

Rows dropped (missing critical fields): 0
Rows after outlier removal: 2919079

Flight dataset cleaned: (2919079, 32)


In [6]:
# Fix date format
weather_df['FL_DATE'] = pd.to_datetime(weather_df['FL_DATE'])

# Remove duplicates
before = weather_df.shape[0]
weather_df = weather_df.drop_duplicates()
print(f"Duplicates removed: {before - weather_df.shape[0]}")

print("Weather dataset cleaned:", weather_df.shape)

Duplicates removed: 0
Weather dataset cleaned: (1495, 5)


In [7]:
print("=== DATA QUALITY REPORT ===")
print("\n--- Flight Dataset ---")
print(f"Completeness: {round((1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100, 2)}%")
print(f"Uniqueness: {round((1 - df.duplicated().sum() / df.shape[0]) * 100, 2)}%")
print(f"Total records: {df.shape[0]}")

print("\n--- Weather Dataset ---")
print(f"Completeness: {round((1 - weather_df.isnull().sum().sum() / (weather_df.shape[0] * weather_df.shape[1])) * 100, 2)}%")
print(f"Uniqueness: {round((1 - weather_df.duplicated().sum() / weather_df.shape[0]) * 100, 2)}%")
print(f"Total records: {weather_df.shape[0]}")

=== DATA QUALITY REPORT ===

--- Flight Dataset ---
Completeness: 96.84%
Uniqueness: 100.0%
Total records: 2919079

--- Weather Dataset ---
Completeness: 100.0%
Uniqueness: 100.0%
Total records: 1495


In [9]:
import os

folders = [
    '/content/drive/MyDrive/DM_Project/Data/Raw',
    '/content/drive/MyDrive/DM_Project/Data/Cleaned',
    '/content/drive/MyDrive/DM_Project/Data/Integrated',
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Created: {folder}")

Created: /content/drive/MyDrive/DM_Project/Data/Raw
Created: /content/drive/MyDrive/DM_Project/Data/Cleaned
Created: /content/drive/MyDrive/DM_Project/Data/Integrated


In [10]:
# Save cleaned data to folders
df.to_csv('/content/drive/MyDrive/DM_Project/Data/Cleaned/flights_cleaned.csv', index=False)
weather_df.to_csv('/content/drive/MyDrive/DM_Project/Data/Cleaned/weather_cleaned.csv', index=False)
print("flights_cleaned.csv saved to Data/Cleaned/")
print("weather_cleaned.csv saved to Data/Cleaned/")

flights_cleaned.csv saved to Data/Cleaned/
weather_cleaned.csv saved to Data/Cleaned/
